In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


In [ ]:
# --- Data Directory Setup (auto-generated) ---
from pathlib import Path
import os

# Resolve data directory relative to workspace root
def _find_data_dir():
    """Find the data directory by walking up from notebook location."""
    candidates = [
        Path.cwd().parent / "data" / "NLP Project 22. - Sentiment Analysis - Restaurant Reviews",
        Path.cwd() / "data" / "NLP Project 22. - Sentiment Analysis - Restaurant Reviews",
        Path(".").resolve().parent / "data" / "NLP Project 22. - Sentiment Analysis - Restaurant Reviews",
    ]
    for c in candidates:
        if c.exists():
            return c
    # Fallback: current directory
    return Path(".")

DATA_DIR = _find_data_dir()
print(f"Data directory: {DATA_DIR}")

In [0]:
# Importing essential libraries
import numpy as np
import pandas as pd

In [0]:
# Loading the dataset
df = pd.read_csv(str(DATA_DIR / 'Restaurant_Reviews.tsv'), delimiter='\t', quoting=3)

In [4]:
df.shape

(1000, 2)

In [5]:
df.columns

Index(['Review', 'Liked'], dtype='object')

In [6]:
df.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


# **Data Preprocessing**

In [7]:
# Importing essential libraries for performing Natural Language Processing on 'Restaurant_Reviews.tsv' dataset
import nltk
import re
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [0]:
# Cleaning the reviews
corpus = []
for i in range(0,1000):

  # Cleaning special character from the reviews
  review = re.sub(pattern='[^a-zA-Z]',repl=' ', string=df['Review'][i])

  # Converting the entire review into lower case
  review = review.lower()

  # Tokenizing the review by words
  review_words = review.split()

  # Removing the stop words
  review_words = [word for word in review_words if not word in set(stopwords.words('english'))]

  # Stemming the words
  ps = PorterStemmer()
  review = [ps.stem(word) for word in review_words]

  # Joining the stemmed words
  review = ' '.join(review)

  # Creating a corpus
  corpus.append(review)

In [9]:
corpus[0:10]

['wow love place',
 'crust good',
 'tasti textur nasti',
 'stop late may bank holiday rick steve recommend love',
 'select menu great price',
 'get angri want damn pho',
 'honeslti tast fresh',
 'potato like rubber could tell made ahead time kept warmer',
 'fri great',
 'great touch']

In [0]:
# Creating the Bag of Words model
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=1500)
X = cv.fit_transform(corpus).toarray()
y = df.iloc[:, 1].values

---
# Standardized ML Pipeline

**Step 1** — LazyPredict: automated baseline comparison of dozens of models  
**Step 2** — PyCaret: automated final pipeline (setup → compare → finalize)


In [ ]:
# ── STEP 1: LazyPredict — Baseline Model Comparison ──
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

_X = X.toarray() if hasattr(X, 'toarray') else np.array(X) if not isinstance(X, np.ndarray) else X
_y = np.array(y).ravel()

X_train, X_test, y_train, y_test = train_test_split(_X, _y, test_size=0.2, random_state=42)

clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = clf.fit(X_train, X_test, y_train, y_test)
print(models)

# ── Metrics Extraction ──
best_model_name = models.sort_values('Accuracy', ascending=False).index[0]
_best_row = models.loc[best_model_name]
lp_accuracy = float(_best_row.get('Accuracy', 0))
lp_f1 = float(_best_row.get('F1 Score', 0))
print(f'\n>>> Best LazyPredict model: {best_model_name}')
print(f'    Accuracy={lp_accuracy:.4f}  F1={lp_f1:.4f}')


In [ ]:
# ── STEP 2: PyCaret — Final Pipeline ──
from pycaret.classification import setup, compare_models, finalize_model, pull

# Prepare DataFrame (sample if needed for speed)
_max_rows, _max_cols = 5000, 2000
_X_sub = _X[:_max_rows, :min(_X.shape[1], _max_cols)]
_y_sub = _y[:_max_rows]

df_ml = pd.DataFrame(_X_sub, columns=[f'f{i}' for i in range(_X_sub.shape[1])])
df_ml['target'] = _y_sub

s = setup(data=df_ml, target='target', session_id=42, verbose=False)
best = compare_models(n_select=1)
pycaret_results = pull()
print(pycaret_results)

final_model = finalize_model(best)
pycaret_model_name = type(best).__name__

# Extract PyCaret metrics
_pc_best = pycaret_results.iloc[0]
pc_accuracy = float(_pc_best.get('Accuracy', 0))
pc_precision = float(_pc_best.get('Prec.', 0))
pc_recall = float(_pc_best.get('Recall', 0))
pc_f1 = float(_pc_best.get('F1', 0))

print(f'\nPyCaret Best: {pycaret_model_name}')
print(f'  Accuracy={pc_accuracy:.4f}  Precision={pc_precision:.4f}  Recall={pc_recall:.4f}  F1={pc_f1:.4f}')


---
## Model Governance — Persistence & Registry

Save trained model, feature vectorizer, metrics, and register in global project registry.


In [ ]:
import json, os
from datetime import datetime
from pathlib import Path
from joblib import dump

project_name = 'restaurant_reviews'
_artifacts_dir = Path('..') / 'artifacts' / project_name
_artifacts_dir.mkdir(parents=True, exist_ok=True)

# Save trained model
dump(final_model, str(_artifacts_dir / 'model.joblib'))

# Save feature vectorizer
dump(cv, str(_artifacts_dir / 'vectorizer.joblib'))

# Save metrics
_metrics = {
    'best_model_lazypredict': best_model_name,
    'pycaret_model': pycaret_model_name,
    'accuracy': pc_accuracy,
    'f1': pc_f1,
    'precision': pc_precision,
    'recall': pc_recall,
    'lp_accuracy': lp_accuracy,
    'lp_f1': lp_f1,
}
with open(str(_artifacts_dir / 'metrics.json'), 'w') as f:
    json.dump(_metrics, f, indent=2)

# Update global registry
_registry_path = Path('..') / 'artifacts' / 'global_registry.json'
_registry_path.parent.mkdir(parents=True, exist_ok=True)
if _registry_path.exists():
    with open(str(_registry_path)) as f:
        _registry = json.load(f)
else:
    _registry = []
_registry = [e for e in _registry if e.get('project') != project_name]
_registry.append({
    'project': project_name,
    'best_model': best_model_name,
    'pycaret_model': pycaret_model_name,
    'accuracy': pc_accuracy,
    'timestamp': datetime.now().isoformat(),
})
with open(str(_registry_path), 'w') as f:
    json.dump(_registry, f, indent=2)

print(f'Artifacts saved to {_artifacts_dir}/')


In [ ]:
# ── Inference Function ──
def predict_text(text):
    """Run inference on a single text input."""
    vec = cv.transform([text])
    return final_model.predict(vec)

print('Inference function ready: predict_text(text)')


In [ ]:
# ── Consistency Checks ──
assert final_model is not None, 'Final model was not created'
assert best_model_name is not None, 'Best model name was not captured'
assert (_artifacts_dir / 'model.joblib').exists(), 'Model file not saved'
assert (_artifacts_dir / 'metrics.json').exists(), 'Metrics file not saved'

# ── Summary ──
print('=' * 50)
print('MODEL GOVERNANCE SUMMARY')
print('=' * 50)
print(f'Project:              {project_name}')
print(f'Best Model (LP):      {best_model_name}')
print(f'Best Model (PyCaret): {pycaret_model_name}')
print(f'Accuracy:             {pc_accuracy:.4f}')
print(f'Precision:            {pc_precision:.4f}')
print(f'Recall:               {pc_recall:.4f}')
print(f'F1 Score:             {pc_f1:.4f}')
print(f'Artifacts:            {_artifacts_dir}/')
print('=' * 50)
